# 03 - Paraphrase COTs

Generate all COT variants using Qwen3-8B as paraphraser:
- **paraphrase_light**: Reword while preserving all steps and numbers
- **paraphrase_heavy**: Compress to just key steps and intermediate results

Also generate Python-based transformations:
- **shuffled_steps**: Randomly reorder COT steps
- **corrupted_numbers**: Replace intermediate numbers with random values

In [ ]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/11-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/JackHopkins/legibility.git", str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
from tqdm import tqdm
from lib.paraphrase import shuffle_steps, corrupt_numbers
from lib.prompts import build_paraphrase_light_messages, build_paraphrase_heavy_messages

## Load Original COTs

In [ ]:
cots = {}
for p in sorted(COT_CACHE.glob("*.json"), key=lambda x: int(x.stem)):
    data = json.loads(p.read_text())
    cots[data["problem_id"]] = data

print(f"Loaded {len(cots)} COTs")
# Filter to only problems with valid COTs
valid_cots = {pid: c for pid, c in cots.items() if c["cot_text"] and c["error"] is None}
print(f"Valid COTs (non-empty, no error): {len(valid_cots)}")

## Load Paraphraser Model (Qwen3-8B)

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

llm = LLM(
    model=PARAPHRASER_MODEL,
    dtype="bfloat16",
    max_model_len=4096,
)
tokenizer = AutoTokenizer.from_pretrained(PARAPHRASER_MODEL)
sampling = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_COT_TOKENS)
print(f"Loaded paraphraser: {PARAPHRASER_MODEL}")

## Generate Light Paraphrases

In [ ]:
condition = "paraphrase_light"
done_ids = {
    int(p.stem.split("_")[-1])
    for p in PARAPHRASE_CACHE.glob(f"{condition}_*.json")
}
todo = [(pid, c) for pid, c in valid_cots.items() if pid not in done_ids]
print(f"[{condition}] {len(done_ids)} done, {len(todo)} remaining")

CHUNK_SIZE = 32

for i in tqdm(range(0, len(todo), CHUNK_SIZE), desc=condition):
    chunk = todo[i : i + CHUNK_SIZE]

    prompts = []
    for pid, cot_data in chunk:
        messages = build_paraphrase_light_messages(cot_data["cot_text"])
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
        prompts.append(prompt)

    outputs = llm.generate(prompts, sampling)

    for (pid, cot_data), output in zip(chunk, outputs):
        paraphrased = output.outputs[0].text.strip()

        result = {
            "problem_id": pid,
            "condition": condition,
            "original_cot": cot_data["cot_text"],
            "paraphrased_cot": paraphrased,
        }
        (PARAPHRASE_CACHE / f"{condition}_{pid}.json").write_text(json.dumps(result))

print(f"Light paraphrasing complete.")

## Generate Heavy Paraphrases

In [ ]:
condition = "paraphrase_heavy"
done_ids = {
    int(p.stem.split("_")[-1])
    for p in PARAPHRASE_CACHE.glob(f"{condition}_*.json")
}
todo = [(pid, c) for pid, c in valid_cots.items() if pid not in done_ids]
print(f"[{condition}] {len(done_ids)} done, {len(todo)} remaining")

for i in tqdm(range(0, len(todo), CHUNK_SIZE), desc=condition):
    chunk = todo[i : i + CHUNK_SIZE]

    prompts = []
    for pid, cot_data in chunk:
        messages = build_paraphrase_heavy_messages(cot_data["cot_text"])
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
        prompts.append(prompt)

    outputs = llm.generate(prompts, sampling)

    for (pid, cot_data), output in zip(chunk, outputs):
        paraphrased = output.outputs[0].text.strip()

        result = {
            "problem_id": pid,
            "condition": condition,
            "original_cot": cot_data["cot_text"],
            "paraphrased_cot": paraphrased,
        }
        (PARAPHRASE_CACHE / f"{condition}_{pid}.json").write_text(json.dumps(result))

print(f"Heavy paraphrasing complete.")

In [ ]:
# Free GPU for model swap in next notebook
del llm
import gc; gc.collect()
import torch; torch.cuda.empty_cache()
print("Paraphraser model freed.")

## Generate Shuffled Steps (Python, no LLM)

In [ ]:
condition = "shuffled_steps"
done_ids = {
    int(p.stem.split("_")[-1])
    for p in PARAPHRASE_CACHE.glob(f"{condition}_*.json")
}
todo = [(pid, c) for pid, c in valid_cots.items() if pid not in done_ids]
print(f"[{condition}] {len(done_ids)} done, {len(todo)} remaining")

for pid, cot_data in tqdm(todo, desc=condition):
    shuffled = shuffle_steps(cot_data["cot_text"], seed=pid)

    result = {
        "problem_id": pid,
        "condition": condition,
        "original_cot": cot_data["cot_text"],
        "paraphrased_cot": shuffled,
    }
    (PARAPHRASE_CACHE / f"{condition}_{pid}.json").write_text(json.dumps(result))

print(f"Shuffled steps complete.")

## Generate Corrupted Numbers (Python, no LLM)

In [ ]:
condition = "corrupted_numbers"
done_ids = {
    int(p.stem.split("_")[-1])
    for p in PARAPHRASE_CACHE.glob(f"{condition}_*.json")
}
todo = [(pid, c) for pid, c in valid_cots.items() if pid not in done_ids]
print(f"[{condition}] {len(done_ids)} done, {len(todo)} remaining")

for pid, cot_data in tqdm(todo, desc=condition):
    corrupted = corrupt_numbers(
        cot_data["cot_text"],
        final_answer=cot_data["gold_answer"],
        seed=pid,
    )

    result = {
        "problem_id": pid,
        "condition": condition,
        "original_cot": cot_data["cot_text"],
        "paraphrased_cot": corrupted,
    }
    (PARAPHRASE_CACHE / f"{condition}_{pid}.json").write_text(json.dumps(result))

print(f"Corrupted numbers complete.")

## Verify Paraphrase Quality

In [ ]:
# Show sample paraphrases for manual inspection
sample_pid = list(valid_cots.keys())[0]

print("=" * 80)
print(f"Problem {sample_pid}")
print("=" * 80)

print("\n--- ORIGINAL COT ---")
print(valid_cots[sample_pid]["cot_text"][:500])

for cond in ["paraphrase_light", "paraphrase_heavy", "shuffled_steps", "corrupted_numbers"]:
    path = PARAPHRASE_CACHE / f"{cond}_{sample_pid}.json"
    if path.exists():
        data = json.loads(path.read_text())
        print(f"\n--- {cond.upper()} ---")
        print(data["paraphrased_cot"][:500])

In [ ]:
# Count paraphrases per condition
for cond in ["paraphrase_light", "paraphrase_heavy", "shuffled_steps", "corrupted_numbers"]:
    count = len(list(PARAPHRASE_CACHE.glob(f"{cond}_*.json")))
    print(f"{cond}: {count} paraphrases")